# 03 — LazyPredict Lider Modeli ve Regression Çıktıları

Bu v5 sürümünde bütün kod hücreleri eğitim amacıyla satır satır açıklanmıştır. Her çalıştırılabilir satırın hemen üzerinde, o satırın görevini özetleyen kısa bir `#` notu bulunur.

Bu notebook yalnızca **modelleme** yapar.

- Model girdisi olarak doğrudan `CHEMBL206_Mordred2D_features.csv` kullanılır.
- Dosya önce Google Drive'dan okunur; bulunamazsa GitHub RAW bağlantısı denenir.
- Ayrı bir Lipinski CSV ile satır filtreleme yapılmaz.
- Feature filtreleme tekrar edilmez.
- Train/test split yalnızca bir kez oluşturulur.
- Median imputer yalnızca eğitim setinde fit edilir.
- LazyPredict model ailelerini karşılaştırmak için kullanılır.
- LazyPredict tarafından eğitilen bütün başarılı model nesneleri Drive'a kaydedilir.
- Performans tablosu bilimsel gösterim yerine dört ondalık basamakla gösterilir.
- LazyPredict sıralamasının ilk satırındaki lider model doğrudan kullanılır.
- LazyPredict sonrasında ayrı bir Random Forest, Extra Trees veya başka ağaç modeli eğitilmez.
- Lider modelin tahmin, metrik, çapraz doğrulama ve grafik çıktıları oluşturulur.
- Lider model, imputer ve split bilgileri Notebook 04 için kaydedilir.


## 1 — Paketler

Bu bölüm, notebookun ihtiyaç duyduğu Python paketlerini kontrol eder. Eksik paketler yalnızca gerektiğinde kurulur; böylece aynı kurulum komutları gereksiz yere tekrar çalıştırılmaz. Hücre tamamlandığında sonraki bölümlerde kullanılacak temel yazılımlar hazır olur.

In [ ]:
# importlib.util paketini kullanılabilir hâle getirir.
import importlib.util
# subprocess paketini kullanılabilir hâle getirir.
import subprocess
# sys paketini kullanılabilir hâle getirir.
import sys

# `PACKAGES` değişkenine bu adımda kullanılacak değeri atar.
PACKAGES = [
    # Çok satırlı veri yapısını veya fonksiyon çağrısını başlatır.
    ("numpy", "numpy"),
    # Çok satırlı veri yapısını veya fonksiyon çağrısını başlatır.
    ("pandas", "pandas"),
    # Çok satırlı veri yapısını veya fonksiyon çağrısını başlatır.
    ("requests", "requests"),
    # Çok satırlı veri yapısını veya fonksiyon çağrısını başlatır.
    ("sklearn", "scikit-learn"),
    # Çok satırlı veri yapısını veya fonksiyon çağrısını başlatır.
    ("joblib", "joblib"),
    # Çok satırlı veri yapısını veya fonksiyon çağrısını başlatır.
    ("matplotlib", "matplotlib"),
    # Çok satırlı veri yapısını veya fonksiyon çağrısını başlatır.
    ("lazypredict", "lazypredict==0.2.16"),
# Önceki çok satırlı yapı veya fonksiyon çağrısını kapatır.
]

# Belirtilen koleksiyondaki öğeleri sırayla işler.
for import_name, pip_name in PACKAGES:
    # Belirtilen koşul doğruysa aşağıdaki kod bloğunu çalıştırır.
    if importlib.util.find_spec(import_name) is None:
        # Pipeline'ın bu aşaması için gerekli Python ifadesini çalıştırır.
        subprocess.check_call(
            # Çok satırlı veri yapısını veya fonksiyon çağrısını başlatır.
            [sys.executable, "-m", "pip", "install", "-q", pip_name]
        # Önceki çok satırlı yapı veya fonksiyon çağrısını kapatır.
        )

# İşlem sonucunu veya kontrol bilgisini ekrana yazdırır.
print("Paket kontrolü tamamlandı.")


## 2 — Kütüphaneler ve ayarlar

Bu bölümde kullanılacak kütüphaneler belleğe alınır ve pipeline boyunca değişmeden kullanılacak temel ayarlar tanımlanır. Target ID, klasör yolları, dosya adları ve tekrar üretilebilirlik değerleri burada merkezi olarak tutulur.

Bu güncellemede ayrıca LazyPredict tarafından eğitilen model nesnelerini kopyalamadan kullanmak, lider modeli çapraz doğrulamada aynı mimariyle değerlendirmek ve Drive dosya adlarını güvenli biçimde üretmek için gerekli araçlar içe aktarılır.


In [ ]:
# Gerekli sınıf veya fonksiyonları ilgili Python modülünden içe aktarır.
from pathlib import Path
# Gerekli sınıf veya fonksiyonları ilgili Python modülünden içe aktarır.
from io import BytesIO
# Gerekli Python paketini kullanılabilir hâle getirir.
import json
# Gerekli Python paketini kullanılabilir hâle getirir.
import re

# Gerekli Python paketini kullanılabilir hâle getirir.
import joblib
# Gerekli Python paketini kullanılabilir hâle getirir.
import matplotlib.pyplot as plt
# Gerekli Python paketini kullanılabilir hâle getirir.
import numpy as np
# Gerekli Python paketini kullanılabilir hâle getirir.
import pandas as pd
# Gerekli Python paketini kullanılabilir hâle getirir.
import requests

# Gerekli sınıf veya fonksiyonları ilgili Python modülünden içe aktarır.
from google.colab import drive
# Gerekli sınıf veya fonksiyonları ilgili Python modülünden içe aktarır.
from IPython.display import display

# Gerekli sınıf veya fonksiyonları ilgili Python modülünden içe aktarır.
from lazypredict.Supervised import LazyRegressor

# Gerekli sınıf veya fonksiyonları ilgili Python modülünden içe aktarır.
from sklearn.base import clone
# Gerekli sınıf veya fonksiyonları ilgili Python modülünden içe aktarır.
from sklearn.impute import SimpleImputer
# Gerekli sınıf veya fonksiyonları ilgili Python modülünden içe aktarır.
from sklearn.metrics import (
    # Çok satırlı yapıdaki bu parametre veya öğeyi tanımlar.
    mean_absolute_error,
    # Çok satırlı yapıdaki bu parametre veya öğeyi tanımlar.
    mean_squared_error,
    # Çok satırlı yapıdaki bu parametre veya öğeyi tanımlar.
    r2_score,
# Önceki çok satırlı yapı veya fonksiyon çağrısını kapatır.
)
# Gerekli sınıf veya fonksiyonları ilgili Python modülünden içe aktarır.
from sklearn.model_selection import (
    # Çok satırlı yapıdaki bu parametre veya öğeyi tanımlar.
    KFold,
    # Çok satırlı yapıdaki bu parametre veya öğeyi tanımlar.
    cross_validate,
    # Çok satırlı yapıdaki bu parametre veya öğeyi tanımlar.
    train_test_split,
# Önceki çok satırlı yapı veya fonksiyon çağrısını kapatır.
)

# Pipeline'ın bu aşaması için gerekli Python ifadesini çalıştırır.
drive.mount("/content/drive", force_remount=False)

# `DRIVE_DIR` değişkenine bu adımda kullanılacak değeri atar.
DRIVE_DIR = Path(
    # Pipeline'ın bu aşaması için gerekli Python ifadesini çalıştırır.
    "/content/drive/MyDrive/MOL_FET_regression_pipeline"
# Önceki çok satırlı yapı veya fonksiyon çağrısını kapatır.
)
# `CONFIG_PATH` değişkenine bu adımda kullanılacak değeri atar.
CONFIG_PATH = DRIVE_DIR / "pipeline_config.json"

# `DEFAULT_TARGET_ID` değişkenine bu adımda kullanılacak değeri atar.
DEFAULT_TARGET_ID = "CHEMBL206"
# `DEFAULT_GITHUB_RAW_BASE` değişkenine bu adımda kullanılacak değeri atar.
DEFAULT_GITHUB_RAW_BASE = (
    # Pipeline'ın bu aşaması için gerekli Python ifadesini çalıştırır.
    "https://raw.githubusercontent.com/"
    # Pipeline'ın bu aşaması için gerekli Python ifadesini çalıştırır.
    "MOL-FEST/MOL-FET_regression/main"
# Önceki çok satırlı yapı veya fonksiyon çağrısını kapatır.
)

# Belirtilen koşul doğruysa aşağıdaki kod bloğunu çalıştırır.
if CONFIG_PATH.exists():
    # `config` değişkenine bu adımda kullanılacak değeri atar.
    config = json.loads(CONFIG_PATH.read_text(encoding="utf-8"))
# Önceki koşullar sağlanmadığında alternatif kod bloğunu çalıştırır.
else:
    # `config` değişkenine bu adımda kullanılacak değeri atar.
    config = {
        # Çok satırlı yapıdaki bu parametre veya öğeyi tanımlar.
        "target_id": DEFAULT_TARGET_ID,
        # Çok satırlı yapıdaki bu parametre veya öğeyi tanımlar.
        "github_raw_base": DEFAULT_GITHUB_RAW_BASE,
        # Çok satırlı yapıdaki bu parametre veya öğeyi tanımlar.
        "feature_filename": f"{DEFAULT_TARGET_ID}_Mordred2D_features.csv",
        # Çok satırlı yapıdaki bu parametre veya öğeyi tanımlar.
        "imputer_filename": f"{DEFAULT_TARGET_ID}_feature_imputer.joblib",
        # Çok satırlı yapıdaki bu parametre veya öğeyi tanımlar.
        "bundle_filename": f"{DEFAULT_TARGET_ID}_model_bundle.json",
    # Önceki çok satırlı yapı veya fonksiyon çağrısını kapatır.
    }

# `TARGET_ID` değişkenine bu adımda kullanılacak değeri atar.
TARGET_ID = config["target_id"]
# `GITHUB_RAW_BASE` değişkenine bu adımda kullanılacak değeri atar.
GITHUB_RAW_BASE = config["github_raw_base"]
# `FEATURE_FILENAME` değişkenine bu adımda kullanılacak değeri atar.
FEATURE_FILENAME = f"{TARGET_ID}_Mordred2D_features.csv"
# `IMPUTER_FILENAME` değişkenine bu adımda kullanılacak değeri atar.
IMPUTER_FILENAME = config["imputer_filename"]
# `BUNDLE_FILENAME` değişkenine bu adımda kullanılacak değeri atar.
BUNDLE_FILENAME = config["bundle_filename"]

# `LEADER_MODEL_FILENAME` değişkenine bu adımda kullanılacak değeri atar.
LEADER_MODEL_FILENAME = (
    # Pipeline'ın bu aşaması için gerekli Python ifadesini çalıştırır.
    f"{TARGET_ID}_LazyPredict_leader_model.joblib"
# Önceki çok satırlı yapı veya fonksiyon çağrısını kapatır.
)
# `LAZY_MODELS_FOLDERNAME` değişkenine bu adımda kullanılacak değeri atar.
LAZY_MODELS_FOLDERNAME = (
    # Pipeline'ın bu aşaması için gerekli Python ifadesini çalıştırır.
    f"{TARGET_ID}_LazyPredict_models"
# Önceki çok satırlı yapı veya fonksiyon çağrısını kapatır.
)
# `LAZY_MODEL_REGISTRY_FILENAME` değişkenine bu adımda kullanılacak değeri atar.
LAZY_MODEL_REGISTRY_FILENAME = (
    # Pipeline'ın bu aşaması için gerekli Python ifadesini çalıştırır.
    f"{TARGET_ID}_LazyPredict_model_registry.csv"
# Önceki çok satırlı yapı veya fonksiyon çağrısını kapatır.
)

# `RANDOM_STATE` değişkenine bu adımda kullanılacak değeri atar.
RANDOM_STATE = 42
# `TEST_SIZE` değişkenine bu adımda kullanılacak değeri atar.
TEST_SIZE = 0.20
# `CV_FOLDS` değişkenine bu adımda kullanılacak değeri atar.
CV_FOLDS = 5

# Pipeline'ın bu aşaması için gerekli Python ifadesini çalıştırır.
pd.set_option(
    # Çok satırlı yapıdaki bu parametre veya öğeyi tanımlar.
    "display.float_format",
    # Çok satırlı yapıdaki bu parametre veya öğeyi tanımlar.
    lambda value: f"{value:.4f}",
# Önceki çok satırlı yapı veya fonksiyon çağrısını kapatır.
)

# Gerekli klasörü mevcut değilse oluşturur.
DRIVE_DIR.mkdir(parents=True, exist_ok=True)

## 3 — Drive öncelikli Mordred2D feature store

Bu bölüm modelleme için gerekli olan `CHEMBL206_Mordred2D_features.csv` dosyasını yükler.

Dosya önce Google Drive içindeki ortak pipeline klasöründe aranır. Drive dosyası bulunamazsa aynı dosya GitHub `data/` klasöründeki RAW bağlantıdan okunur.

Bu notebook ayrı bir Lipinski CSV kullanmaz. Target, metadata ve filtrelenmiş 2D Mordred descriptorları aynı model-ready feature store içinde yer alır.


In [ ]:
# `load_csv` adlı yardımcı fonksiyonu tanımlar.
def load_csv(
    # Çok satırlı yapıdaki bu parametre veya öğeyi tanımlar.
    drive_filename,
    # Çok satırlı yapıdaki bu parametre veya öğeyi tanımlar.
    github_relative_path,
# Önceki çok satırlı yapı veya fonksiyon çağrısını kapatır.
):
    # `drive_path` değişkenine bu adımda kullanılacak değeri atar.
    drive_path = (
        # Pipeline'ın bu aşaması için gerekli Python ifadesini çalıştırır.
        DRIVE_DIR / drive_filename
    # Önceki çok satırlı yapı veya fonksiyon çağrısını kapatır.
    )

    # Belirtilen koşul doğruysa aşağıdaki kod bloğunu çalıştırır.
    if drive_path.exists():
        # Kontrol veya sonuç bilgisini ekrana yazdırır.
        print(
            # Çok satırlı yapıdaki bu parametre veya öğeyi tanımlar.
            "Girdi kaynağı: Google Drive",
            # Çok satırlı yapıdaki bu parametre veya öğeyi tanımlar.
            drive_filename,
        # Önceki çok satırlı yapı veya fonksiyon çağrısını kapatır.
        )

        # Fonksiyon sonucunu çağrıldığı yere döndürür.
        return pd.read_csv(
            # Çok satırlı yapıdaki bu parametre veya öğeyi tanımlar.
            drive_path,
            # Çok satırlı yapıdaki bu parametre veya öğeyi tanımlar.
            low_memory=False,
        # Önceki çok satırlı yapı veya fonksiyon çağrısını kapatır.
        )

    # `github_url` değişkenine bu adımda kullanılacak değeri atar.
    github_url = (
        # Pipeline'ın bu aşaması için gerekli Python ifadesini çalıştırır.
        f"{GITHUB_RAW_BASE}/"
        # Pipeline'ın bu aşaması için gerekli Python ifadesini çalıştırır.
        f"{github_relative_path}"
    # Önceki çok satırlı yapı veya fonksiyon çağrısını kapatır.
    )

    # GitHub RAW bağlantısına HTTP isteği gönderir.
    response = requests.get(
        # Çok satırlı yapıdaki bu parametre veya öğeyi tanımlar.
        github_url,
        # Çok satırlı yapıdaki bu parametre veya öğeyi tanımlar.
        timeout=120,
    # Önceki çok satırlı yapı veya fonksiyon çağrısını kapatır.
    )

    # HTTP yanıtında hata varsa işlemi açıklayıcı hatayla durdurur.
    response.raise_for_status()

    # Kontrol veya sonuç bilgisini ekrana yazdırır.
    print(
        # Çok satırlı yapıdaki bu parametre veya öğeyi tanımlar.
        "Girdi kaynağı: GitHub",
        # Çok satırlı yapıdaki bu parametre veya öğeyi tanımlar.
        github_relative_path,
    # Önceki çok satırlı yapı veya fonksiyon çağrısını kapatır.
    )

    # Fonksiyon sonucunu çağrıldığı yere döndürür.
    return pd.read_csv(
        # Çok satırlı yapıdaki bu parametre veya öğeyi tanımlar.
        BytesIO(response.content),
        # Çok satırlı yapıdaki bu parametre veya öğeyi tanımlar.
        low_memory=False,
    # Önceki çok satırlı yapı veya fonksiyon çağrısını kapatır.
    )


# `feature_df` değişkenine bu adımda kullanılacak değeri atar.
feature_df = load_csv(
    # Çok satırlı yapıdaki bu parametre veya öğeyi tanımlar.
    FEATURE_FILENAME,
    # Çok satırlı yapıdaki bu parametre veya öğeyi tanımlar.
    f"data/{FEATURE_FILENAME}",
# Önceki çok satırlı yapı veya fonksiyon çağrısını kapatır.
)

# `required_feature_columns` değişkenine bu adımda kullanılacak değeri atar.
required_feature_columns = {
    # Çok satırlı yapıdaki bu parametre veya öğeyi tanımlar.
    "parent_smiles",
    # Çok satırlı yapıdaki bu parametre veya öğeyi tanımlar.
    "pStandard",
# Önceki çok satırlı yapı veya fonksiyon çağrısını kapatır.
}

# `missing_feature_columns` değişkenine bu adımda kullanılacak değeri atar.
missing_feature_columns = (
    # Pipeline'ın bu aşaması için gerekli Python ifadesini çalıştırır.
    required_feature_columns
    # Pipeline'ın bu aşaması için gerekli Python ifadesini çalıştırır.
    .difference(
        # Pipeline'ın bu aşaması için gerekli Python ifadesini çalıştırır.
        feature_df.columns
    # Önceki çok satırlı yapı veya fonksiyon çağrısını kapatır.
    )
# Önceki çok satırlı yapı veya fonksiyon çağrısını kapatır.
)

# Belirtilen koşul doğruysa aşağıdaki kod bloğunu çalıştırır.
if missing_feature_columns:
    # Beklenen koşul sağlanmadığında açıklayıcı hata oluşturur.
    raise KeyError(
        # Pipeline'ın bu aşaması için gerekli Python ifadesini çalıştırır.
        "Feature store içinde eksik sütunlar: "
        # Pipeline'ın bu aşaması için gerekli Python ifadesini çalıştırır.
        f"{sorted(missing_feature_columns)}"
    # Önceki çok satırlı yapı veya fonksiyon çağrısını kapatır.
    )

# Kontrol veya sonuç bilgisini ekrana yazdırır.
print(
    # Çok satırlı yapıdaki bu parametre veya öğeyi tanımlar.
    "Kullanılan model girdisi:",
    # Çok satırlı yapıdaki bu parametre veya öğeyi tanımlar.
    FEATURE_FILENAME,
# Önceki çok satırlı yapı veya fonksiyon çağrısını kapatır.
)

# Kontrol veya sonuç bilgisini ekrana yazdırır.
print(
    # Çok satırlı yapıdaki bu parametre veya öğeyi tanımlar.
    "Feature store boyutu:",
    # Çok satırlı yapıdaki bu parametre veya öğeyi tanımlar.
    feature_df.shape,
# Önceki çok satırlı yapı veya fonksiyon çağrısını kapatır.
)

## 4 — Target, metadata ve Mordred feature sütunlarının ayrılması

Bu bölüm `CHEMBL206_Mordred2D_features.csv` içindeki target, metadata ve model featurelarını birbirinden ayırır.

Modellemeye, sayısal `pStandard` değeri bulunan bütün satırlar alınır. Ayrı bir Lipinski dosyasıyla satır eşleştirme veya ek filtreleme yapılmaz.

Feature store içindeki orijinal satır indeksleri korunur; böylece Notebook 04 aynı train ve test kayıtlarını bundle üzerinden yeniden kullanabilir.


In [ ]:
# `TARGET_COLUMN` değişkenine bu adımda kullanılacak değeri atar.
TARGET_COLUMN = "pStandard"

# `metadata_candidates` değişkenine bu adımda kullanılacak değeri atar.
metadata_candidates = [
    # Çok satırlı yapıdaki bu parametre veya öğeyi tanımlar.
    "parent_chembl_id",
    # Çok satırlı yapıdaki bu parametre veya öğeyi tanımlar.
    "parent_smiles",
    # Çok satırlı yapıdaki bu parametre veya öğeyi tanımlar.
    "pStandard",
    # Çok satırlı yapıdaki bu parametre veya öğeyi tanımlar.
    "pStandard_mean",
    # Çok satırlı yapıdaki bu parametre veya öğeyi tanımlar.
    "pStandard_std",
    # Çok satırlı yapıdaki bu parametre veya öğeyi tanımlar.
    "pStandard_range",
    # Çok satırlı yapıdaki bu parametre veya öğeyi tanımlar.
    "n_measurements",
    # Çok satırlı yapıdaki bu parametre veya öğeyi tanımlar.
    "n_assays",
    # Çok satırlı yapıdaki bu parametre veya öğeyi tanımlar.
    "activity_conflict",
    # Çok satırlı yapıdaki bu parametre veya öğeyi tanımlar.
    "MolWt",
    # Çok satırlı yapıdaki bu parametre veya öğeyi tanımlar.
    "MolLogP",
    # Çok satırlı yapıdaki bu parametre veya öğeyi tanımlar.
    "HBD",
    # Çok satırlı yapıdaki bu parametre veya öğeyi tanımlar.
    "HBA",
    # Çok satırlı yapıdaki bu parametre veya öğeyi tanımlar.
    "Lipinski_violations",
    # Çok satırlı yapıdaki bu parametre veya öğeyi tanımlar.
    "passes_Lipinski",
# Önceki çok satırlı yapı veya fonksiyon çağrısını kapatır.
]

# `metadata_columns` değişkenine bu adımda kullanılacak değeri atar.
metadata_columns = [
    # Pipeline'ın bu aşaması için gerekli Python ifadesini çalıştırır.
    column
    # Pipeline'ın bu aşaması için gerekli Python ifadesini çalıştırır.
    for column in metadata_candidates
    # Belirtilen koşul doğruysa aşağıdaki kod bloğunu çalıştırır.
    if column in feature_df.columns
# Önceki çok satırlı yapı veya fonksiyon çağrısını kapatır.
]

# `feature_columns` değişkenine bu adımda kullanılacak değeri atar.
feature_columns = [
    # Pipeline'ın bu aşaması için gerekli Python ifadesini çalıştırır.
    column
    # Pipeline'ın bu aşaması için gerekli Python ifadesini çalıştırır.
    for column in feature_df.columns
    # Belirtilen koşul doğruysa aşağıdaki kod bloğunu çalıştırır.
    if column not in metadata_columns
# Önceki çok satırlı yapı veya fonksiyon çağrısını kapatır.
]

# Belirtilen koşul doğruysa aşağıdaki kod bloğunu çalıştırır.
if TARGET_COLUMN not in feature_df.columns:
    # Beklenen koşul sağlanmadığında açıklayıcı hata oluşturur.
    raise KeyError(
        # Pipeline'ın bu aşaması için gerekli Python ifadesini çalıştırır.
        f"Target sütunu bulunamadı: {TARGET_COLUMN}"
    # Önceki çok satırlı yapı veya fonksiyon çağrısını kapatır.
    )

# Belirtilen koşul doğruysa aşağıdaki kod bloğunu çalıştırır.
if not feature_columns:
    # Beklenen koşul sağlanmadığında açıklayıcı hata oluşturur.
    raise ValueError(
        # Pipeline'ın bu aşaması için gerekli Python ifadesini çalıştırır.
        "Model featureı bulunamadı."
    # Önceki çok satırlı yapı veya fonksiyon çağrısını kapatır.
    )

# `X` değişkenine bu adımda kullanılacak değeri atar.
X = (
    # Pipeline'ın bu aşaması için gerekli Python ifadesini çalıştırır.
    feature_df[feature_columns]
    # Belirtilen dönüşümü ilgili DataFrame sütunlarına uygular.
    .apply(
        # Çok satırlı yapıdaki bu parametre veya öğeyi tanımlar.
        pd.to_numeric,
        # Çok satırlı yapıdaki bu parametre veya öğeyi tanımlar.
        errors="coerce",
    # Önceki çok satırlı yapı veya fonksiyon çağrısını kapatır.
    )
    # Sonsuz değerleri güvenli biçimde eksik değerlerle değiştirir.
    .replace(
        # Çok satırlı veri yapısını veya fonksiyon çağrısını başlatır.
        [
            # Çok satırlı yapıdaki bu parametre veya öğeyi tanımlar.
            np.inf,
            # Çok satırlı yapıdaki bu parametre veya öğeyi tanımlar.
            -np.inf,
        # Önceki çok satırlı yapı veya fonksiyon çağrısını kapatır.
        ],
        # Çok satırlı yapıdaki bu parametre veya öğeyi tanımlar.
        np.nan,
    # Önceki çok satırlı yapı veya fonksiyon çağrısını kapatır.
    )
# Önceki çok satırlı yapı veya fonksiyon çağrısını kapatır.
)

# `y` değişkenine bu adımda kullanılacak değeri atar.
y = pd.to_numeric(
    # Çok satırlı yapıdaki bu parametre veya öğeyi tanımlar.
    feature_df[TARGET_COLUMN],
    # Çok satırlı yapıdaki bu parametre veya öğeyi tanımlar.
    errors="coerce",
# Önceki çok satırlı yapı veya fonksiyon çağrısını kapatır.
)

# Geçerli target değeri bulunan satırları belirler.
valid_target_mask = y.notna()

# Modellemeye alınacak geçerli satırların indekslerini seçer.
modeling_indices = np.flatnonzero(
    # pandas nesnesini NumPy dizisine dönüştürür.
    valid_target_mask.to_numpy()
# Önceki çok satırlı yapı veya fonksiyon çağrısını kapatır.
)

# Belirtilen koşul doğruysa aşağıdaki kod bloğunu çalıştırır.
if len(modeling_indices) < 10:
    # Beklenen koşul sağlanmadığında açıklayıcı hata oluşturur.
    raise ValueError(
        # Pipeline'ın bu aşaması için gerekli Python ifadesini çalıştırır.
        "Target kontrolünden sonra "
        # Pipeline'ın bu aşaması için gerekli Python ifadesini çalıştırır.
        "en az 10 molekül gereklidir."
    # Önceki çok satırlı yapı veya fonksiyon çağrısını kapatır.
    )

# Kontrol veya sonuç bilgisini ekrana yazdırır.
print(
    # Çok satırlı yapıdaki bu parametre veya öğeyi tanımlar.
    "Modellemeye alınan molekül:",
    # Çok satırlı yapıdaki bu parametre veya öğeyi tanımlar.
    len(modeling_indices),
# Önceki çok satırlı yapı veya fonksiyon çağrısını kapatır.
)

# Kontrol veya sonuç bilgisini ekrana yazdırır.
print(
    # Çok satırlı yapıdaki bu parametre veya öğeyi tanımlar.
    "Feature sayısı:",
    # Çok satırlı yapıdaki bu parametre veya öğeyi tanımlar.
    len(feature_columns),
# Önceki çok satırlı yapı veya fonksiyon çağrısını kapatır.
)

## 5 — Train/test split ve eğitim tabanlı imputasyon

Bu bölüm `CHEMBL206_Mordred2D_features.csv` içindeki geçerli molekül indekslerini train ve test olarak ayırır.

İndeksler yeniden numaralandırılmaz. Böylece kaydedilen split, Notebook 04 tarafından aynı feature store üzerinde doğrudan kullanılabilir.

Median imputer yalnızca train setinde fit edilir ve aynı dönüşüm test setine uygulanır.


In [ ]:
# Lipinski filtreli molekül indekslerini train ve test olarak ayırır.
train_indices, test_indices = train_test_split(
    # Çok satırlı yapıdaki bu parametre veya öğeyi tanımlar.
    modeling_indices,
    # Çok satırlı yapıdaki bu parametre veya öğeyi tanımlar.
    test_size=TEST_SIZE,
    # Çok satırlı yapıdaki bu parametre veya öğeyi tanımlar.
    random_state=RANDOM_STATE,
    # Çok satırlı yapıdaki bu parametre veya öğeyi tanımlar.
    shuffle=True,
# Önceki çok satırlı yapı veya fonksiyon çağrısını kapatır.
)

# `X_train_raw` değişkenine bu adımda kullanılacak değeri atar.
X_train_raw = X.iloc[
    # Pipeline'ın bu aşaması için gerekli Python ifadesini çalıştırır.
    train_indices
# Önceki çok satırlı yapı veya fonksiyon çağrısını kapatır.
].copy()

# `X_test_raw` değişkenine bu adımda kullanılacak değeri atar.
X_test_raw = X.iloc[
    # Pipeline'ın bu aşaması için gerekli Python ifadesini çalıştırır.
    test_indices
# Önceki çok satırlı yapı veya fonksiyon çağrısını kapatır.
].copy()

# `y_train` değişkenine bu adımda kullanılacak değeri atar.
y_train = y.iloc[
    # Pipeline'ın bu aşaması için gerekli Python ifadesini çalıştırır.
    train_indices
# Önceki çok satırlı yapı veya fonksiyon çağrısını kapatır.
].copy()

# `y_test` değişkenine bu adımda kullanılacak değeri atar.
y_test = y.iloc[
    # Pipeline'ın bu aşaması için gerekli Python ifadesini çalıştırır.
    test_indices
# Önceki çok satırlı yapı veya fonksiyon çağrısını kapatır.
].copy()

# `imputer` değişkenine bu adımda kullanılacak değeri atar.
imputer = SimpleImputer(
    # `strategy` değişkenine bu adımda kullanılacak değeri atar.
    strategy="median"
# Önceki çok satırlı yapı veya fonksiyon çağrısını kapatır.
)

# `X_train` değişkenine bu adımda kullanılacak değeri atar.
X_train = pd.DataFrame(
    # Pipeline'ın bu aşaması için gerekli Python ifadesini çalıştırır.
    imputer.fit_transform(
        # Pipeline'ın bu aşaması için gerekli Python ifadesini çalıştırır.
        X_train_raw
    # Önceki çok satırlı yapı veya fonksiyon çağrısını kapatır.
    ),
    # Çok satırlı yapıdaki bu parametre veya öğeyi tanımlar.
    columns=feature_columns,
    # Çok satırlı yapıdaki bu parametre veya öğeyi tanımlar.
    index=X_train_raw.index,
# Önceki çok satırlı yapı veya fonksiyon çağrısını kapatır.
)

# `X_test` değişkenine bu adımda kullanılacak değeri atar.
X_test = pd.DataFrame(
    # Pipeline'ın bu aşaması için gerekli Python ifadesini çalıştırır.
    imputer.transform(
        # Pipeline'ın bu aşaması için gerekli Python ifadesini çalıştırır.
        X_test_raw
    # Önceki çok satırlı yapı veya fonksiyon çağrısını kapatır.
    ),
    # Çok satırlı yapıdaki bu parametre veya öğeyi tanımlar.
    columns=feature_columns,
    # Çok satırlı yapıdaki bu parametre veya öğeyi tanımlar.
    index=X_test_raw.index,
# Önceki çok satırlı yapı veya fonksiyon çağrısını kapatır.
)

# Belirtilen koşul doğruysa aşağıdaki kod bloğunu çalıştırır.
if not np.isfinite(
    # pandas nesnesini NumPy dizisine dönüştürür.
    X_train.to_numpy()
# Önceki çok satırlı yapı veya fonksiyon çağrısını kapatır.
).all():
    # Beklenen koşul sağlanmadığında açıklayıcı hata oluşturur.
    raise ValueError(
        # Pipeline'ın bu aşaması için gerekli Python ifadesini çalıştırır.
        "Train matrisinde geçersiz değer kaldı."
    # Önceki çok satırlı yapı veya fonksiyon çağrısını kapatır.
    )

# Belirtilen koşul doğruysa aşağıdaki kod bloğunu çalıştırır.
if not np.isfinite(
    # pandas nesnesini NumPy dizisine dönüştürür.
    X_test.to_numpy()
# Önceki çok satırlı yapı veya fonksiyon çağrısını kapatır.
).all():
    # Beklenen koşul sağlanmadığında açıklayıcı hata oluşturur.
    raise ValueError(
        # Pipeline'ın bu aşaması için gerekli Python ifadesini çalıştırır.
        "Test matrisinde geçersiz değer kaldı."
    # Önceki çok satırlı yapı veya fonksiyon çağrısını kapatır.
    )

# Kontrol veya sonuç bilgisini ekrana yazdırır.
print(
    # Çok satırlı yapıdaki bu parametre veya öğeyi tanımlar.
    "Train:",
    # Çok satırlı yapıdaki bu parametre veya öğeyi tanımlar.
    X_train.shape,
# Önceki çok satırlı yapı veya fonksiyon çağrısını kapatır.
)

# Kontrol veya sonuç bilgisini ekrana yazdırır.
print(
    # Çok satırlı yapıdaki bu parametre veya öğeyi tanımlar.
    "Test:",
    # Çok satırlı yapıdaki bu parametre veya öğeyi tanımlar.
    X_test.shape,
# Önceki çok satırlı yapı veya fonksiyon çağrısını kapatır.
)

## 6 — LazyPredict benchmark

Bu bölüm aynı train ve test seti üzerinde LazyPredict regression benchmarkını bir kez çalıştırır.

LazyPredict sonuç tablosundaki sayısal değerler dört ondalık basamakla gösterilir. Böylece `7e-01` veya `2e-10` gibi bilimsel gösterimler yerine `0.7000` ve `0.0000` biçiminde daha kolay okunabilir değerler elde edilir.

Bu hücre yalnızca benchmarkı çalıştırır; lider model seçimi ve model nesnelerinin alınması bir sonraki hücrede yapılır.


In [ ]:
# `lazy_regressor` değişkenine bu adımda kullanılacak değeri atar.
lazy_regressor = LazyRegressor(
    # Çok satırlı yapıdaki bu parametre veya öğeyi tanımlar.
    verbose=0,
    # Çok satırlı yapıdaki bu parametre veya öğeyi tanımlar.
    ignore_warnings=True,
    # Çok satırlı yapıdaki bu parametre veya öğeyi tanımlar.
    custom_metric=None,
    # Çok satırlı yapıdaki bu parametre veya öğeyi tanımlar.
    predictions=True,
    # Çok satırlı yapıdaki bu parametre veya öğeyi tanımlar.
    random_state=RANDOM_STATE,
# Önceki çok satırlı yapı veya fonksiyon çağrısını kapatır.
)

# Modeli veya dönüştürücüyü eğitim verisi üzerinde öğrenir.
lazy_models, lazy_predictions = lazy_regressor.fit(
    # Çok satırlı yapıdaki bu parametre veya öğeyi tanımlar.
    X_train,
    # Çok satırlı yapıdaki bu parametre veya öğeyi tanımlar.
    X_test,
    # Çok satırlı yapıdaki bu parametre veya öğeyi tanımlar.
    y_train,
    # Çok satırlı yapıdaki bu parametre veya öğeyi tanımlar.
    y_test,
# Önceki çok satırlı yapı veya fonksiyon çağrısını kapatır.
)

# Belirtilen koşul doğruysa aşağıdaki kod bloğunu çalıştırır.
if lazy_models.empty:
    # Beklenen koşul sağlanmadığında açıklayıcı hata oluşturur.
    raise RuntimeError("LazyPredict sonuç üretmedi.")

# `lazy_models_display` değişkenine bu adımda kullanılacak değeri atar.
lazy_models_display = lazy_models.copy()

# `numeric_result_columns` değişkenine bu adımda kullanılacak değeri atar.
numeric_result_columns = (
    # Pipeline'ın bu aşaması için gerekli Python ifadesini çalıştırır.
    lazy_models_display
    # Pipeline'ın bu aşaması için gerekli Python ifadesini çalıştırır.
    .select_dtypes(include=[np.number])
    # Pipeline'ın bu aşaması için gerekli Python ifadesini çalıştırır.
    .columns
# Önceki çok satırlı yapı veya fonksiyon çağrısını kapatır.
)

# Pipeline'ın bu aşaması için gerekli Python ifadesini çalıştırır.
lazy_models_display[numeric_result_columns] = (
    # Pipeline'ın bu aşaması için gerekli Python ifadesini çalıştırır.
    lazy_models_display[numeric_result_columns]
    # Pipeline'ın bu aşaması için gerekli Python ifadesini çalıştırır.
    .round(4)
# Önceki çok satırlı yapı veya fonksiyon çağrısını kapatır.
)

# Tabloyu notebook ekranında okunabilir biçimde gösterir.
display(lazy_models_display.head(20))

## 7 — LazyPredict model nesneleri ve lider model

LazyPredict benchmarkı sırasında başarılı biçimde eğitilen model nesneleri `provide_models` yöntemiyle alınır. Bu yöntem, benchmark sonrasında bellekte bulunan eğitilmiş model nesnelerine erişmek için kullanılır.

Performans tablosunun ilk satırındaki model lider kabul edilir. Ayrı bir ağaç modeli kurulmaz veya yeniden seçilmez. Train ve test tahminleri doğrudan bu lider LazyPredict modeliyle üretilir.


In [ ]:
# `lazy_model_dictionary` değişkenine bu adımda kullanılacak değeri atar.
lazy_model_dictionary = lazy_regressor.provide_models(
    # Çok satırlı yapıdaki bu parametre veya öğeyi tanımlar.
    X_train,
    # Çok satırlı yapıdaki bu parametre veya öğeyi tanımlar.
    X_test,
    # Çok satırlı yapıdaki bu parametre veya öğeyi tanımlar.
    y_train,
    # Çok satırlı yapıdaki bu parametre veya öğeyi tanımlar.
    y_test,
# Önceki çok satırlı yapı veya fonksiyon çağrısını kapatır.
)

# `leader_model_name` değişkenine bu adımda kullanılacak değeri atar.
leader_model_name = str(
    # Pipeline'ın bu aşaması için gerekli Python ifadesini çalıştırır.
    lazy_models.index[0]
# Önceki çok satırlı yapı veya fonksiyon çağrısını kapatır.
)

# Belirtilen koşul doğruysa aşağıdaki kod bloğunu çalıştırır.
if leader_model_name not in lazy_model_dictionary:
    # Beklenen koşul sağlanmadığında açıklayıcı hata oluşturur.
    raise KeyError(
        # Pipeline'ın bu aşaması için gerekli Python ifadesini çalıştırır.
        f"Lider model nesnesi bulunamadı: {leader_model_name}"
    # Önceki çok satırlı yapı veya fonksiyon çağrısını kapatır.
    )

# `leader_model` değişkenine bu adımda kullanılacak değeri atar.
leader_model = lazy_model_dictionary[
    # Pipeline'ın bu aşaması için gerekli Python ifadesini çalıştırır.
    leader_model_name
# Önceki çok satırlı yapı veya fonksiyon çağrısını kapatır.
]

# Kaydedilmiş veya eğitilmiş modelle tahmin üretir.
train_prediction = leader_model.predict(
    # Pipeline'ın bu aşaması için gerekli Python ifadesini çalıştırır.
    X_train
# Önceki çok satırlı yapı veya fonksiyon çağrısını kapatır.
)

# Kaydedilmiş veya eğitilmiş modelle tahmin üretir.
test_prediction = leader_model.predict(
    # Pipeline'ın bu aşaması için gerekli Python ifadesini çalıştırır.
    X_test
# Önceki çok satırlı yapı veya fonksiyon çağrısını kapatır.
)

# `leader_model_parameters` değişkenine bu adımda kullanılacak değeri atar.
leader_model_parameters = {
    # Pipeline'ın bu aşaması için gerekli Python ifadesini çalıştırır.
    key: str(value)
    # Koleksiyondaki öğeleri sırayla işler.
    for key, value in leader_model.get_params(
        # `deep` değişkenine bu adımda kullanılacak değeri atar.
        deep=False
    # Önceki çok satırlı yapı veya fonksiyon çağrısını kapatır.
    ).items()
# Önceki çok satırlı yapı veya fonksiyon çağrısını kapatır.
}

# Kontrol veya sonuç bilgisini ekrana yazdırır.
print("LazyPredict lider modeli:", leader_model_name)

## 8 — Lider model holdout ve 5-fold CV

Bu bölüm yalnızca LazyPredict lider modelini değerlendirir.

Holdout tablosunda train ve test için R², RMSE ve MAE değerleri dört ondalık basamakla gösterilir.

Beş katlı çapraz doğrulamada lider model artık dışarıdan bir `SimpleImputer` pipeline'ı içine yerleştirilmez. LazyPredict lider modeli kendi içinde sütun adlarıyla çalışan bir preprocessing pipeline'ı içerebilir. Dış imputer DataFrame'i NumPy dizisine çevirdiğinde bu sütun adları kaybolduğu için önceki sürüm hata veriyordu.

Güncel sürümde lider model doğrudan klonlanır ve sütun adlarını koruyan `X_train` DataFrame'i üzerinde çapraz doğrulanır. Başka bir model ailesi oluşturulmaz.


In [ ]:
# `metric_row` adlı yardımcı fonksiyonu tanımlar.
def metric_row(
    # Çok satırlı yapıdaki bu parametre veya öğeyi tanımlar.
    name,
    # Çok satırlı yapıdaki bu parametre veya öğeyi tanımlar.
    actual,
    # Çok satırlı yapıdaki bu parametre veya öğeyi tanımlar.
    predicted,
# Önceki çok satırlı yapı veya fonksiyon çağrısını kapatır.
):
    # Fonksiyon sonucunu çağrıldığı yere döndürür.
    return {
        # Çok satırlı yapıdaki bu parametre veya öğeyi tanımlar.
        "set": name,
        # Pipeline'ın bu aşaması için gerekli Python ifadesini çalıştırır.
        "R2": r2_score(
            # Çok satırlı yapıdaki bu parametre veya öğeyi tanımlar.
            actual,
            # Çok satırlı yapıdaki bu parametre veya öğeyi tanımlar.
            predicted,
        # Önceki çok satırlı yapı veya fonksiyon çağrısını kapatır.
        ),
        # Pipeline'ın bu aşaması için gerekli Python ifadesini çalıştırır.
        "RMSE": np.sqrt(
            # Pipeline'ın bu aşaması için gerekli Python ifadesini çalıştırır.
            mean_squared_error(
                # Çok satırlı yapıdaki bu parametre veya öğeyi tanımlar.
                actual,
                # Çok satırlı yapıdaki bu parametre veya öğeyi tanımlar.
                predicted,
            # Önceki çok satırlı yapı veya fonksiyon çağrısını kapatır.
            )
        # Önceki çok satırlı yapı veya fonksiyon çağrısını kapatır.
        ),
        # Pipeline'ın bu aşaması için gerekli Python ifadesini çalıştırır.
        "MAE": mean_absolute_error(
            # Çok satırlı yapıdaki bu parametre veya öğeyi tanımlar.
            actual,
            # Çok satırlı yapıdaki bu parametre veya öğeyi tanımlar.
            predicted,
        # Önceki çok satırlı yapı veya fonksiyon çağrısını kapatır.
        ),
    # Önceki çok satırlı yapı veya fonksiyon çağrısını kapatır.
    }


# `metrics` değişkenine bu adımda kullanılacak değeri atar.
metrics = pd.DataFrame(
    # Çok satırlı veri yapısını veya fonksiyon çağrısını başlatır.
    [
        # Pipeline'ın bu aşaması için gerekli Python ifadesini çalıştırır.
        metric_row(
            # Çok satırlı yapıdaki bu parametre veya öğeyi tanımlar.
            "Train",
            # Çok satırlı yapıdaki bu parametre veya öğeyi tanımlar.
            y_train,
            # Çok satırlı yapıdaki bu parametre veya öğeyi tanımlar.
            train_prediction,
        # Önceki çok satırlı yapı veya fonksiyon çağrısını kapatır.
        ),
        # Pipeline'ın bu aşaması için gerekli Python ifadesini çalıştırır.
        metric_row(
            # Çok satırlı yapıdaki bu parametre veya öğeyi tanımlar.
            "Test",
            # Çok satırlı yapıdaki bu parametre veya öğeyi tanımlar.
            y_test,
            # Çok satırlı yapıdaki bu parametre veya öğeyi tanımlar.
            test_prediction,
        # Önceki çok satırlı yapı veya fonksiyon çağrısını kapatır.
        ),
    # Önceki çok satırlı yapı veya fonksiyon çağrısını kapatır.
    ]
# Önceki çok satırlı yapı veya fonksiyon çağrısını kapatır.
)

# `metrics_display` değişkenine bu adımda kullanılacak değeri atar.
metrics_display = metrics.copy()

# `metric_numeric_columns` değişkenine bu adımda kullanılacak değeri atar.
metric_numeric_columns = [
    # Çok satırlı yapıdaki bu parametre veya öğeyi tanımlar.
    "R2",
    # Çok satırlı yapıdaki bu parametre veya öğeyi tanımlar.
    "RMSE",
    # Çok satırlı yapıdaki bu parametre veya öğeyi tanımlar.
    "MAE",
# Önceki çok satırlı yapı veya fonksiyon çağrısını kapatır.
]

# Pipeline'ın bu aşaması için gerekli Python ifadesini çalıştırır.
metrics_display[
    # Pipeline'ın bu aşaması için gerekli Python ifadesini çalıştırır.
    metric_numeric_columns
# Önceki çok satırlı yapı veya fonksiyon çağrısını kapatır.
] = (
    # Pipeline'ın bu aşaması için gerekli Python ifadesini çalıştırır.
    metrics_display[
        # Pipeline'ın bu aşaması için gerekli Python ifadesini çalıştırır.
        metric_numeric_columns
    # Önceki çok satırlı yapı veya fonksiyon çağrısını kapatır.
    ]
    # Sayısal sonuçları okunabilir ondalık basamak sayısına yuvarlar.
    .round(4)
# Önceki çok satırlı yapı veya fonksiyon çağrısını kapatır.
)

# `cv` değişkenine bu adımda kullanılacak değeri atar.
cv = KFold(
    # Çok satırlı yapıdaki bu parametre veya öğeyi tanımlar.
    n_splits=CV_FOLDS,
    # Çok satırlı yapıdaki bu parametre veya öğeyi tanımlar.
    shuffle=True,
    # Çok satırlı yapıdaki bu parametre veya öğeyi tanımlar.
    random_state=RANDOM_STATE,
# Önceki çok satırlı yapı veya fonksiyon çağrısını kapatır.
)

# LazyPredict lider modelini beş katlı çapraz doğrulamayla değerlendirir.
cv_result = cross_validate(
    # Lider modelin aynı ayarlara sahip fakat eğitilmemiş bir kopyasını oluşturur.
    clone(
        # Pipeline'ın bu aşaması için gerekli Python ifadesini çalıştırır.
        leader_model
    # Önceki çok satırlı yapı veya fonksiyon çağrısını kapatır.
    ),
    # Çok satırlı yapıdaki bu parametre veya öğeyi tanımlar.
    X_train,
    # Çok satırlı yapıdaki bu parametre veya öğeyi tanımlar.
    y_train,
    # Çok satırlı yapıdaki bu parametre veya öğeyi tanımlar.
    cv=cv,
    # `scoring` değişkenine bu adımda kullanılacak değeri atar.
    scoring={
        # Çok satırlı yapıdaki bu parametre veya öğeyi tanımlar.
        "R2": "r2",
        # Çok satırlı yapıdaki bu parametre veya öğeyi tanımlar.
        "RMSE": "neg_root_mean_squared_error",
        # Çok satırlı yapıdaki bu parametre veya öğeyi tanımlar.
        "MAE": "neg_mean_absolute_error",
    # Önceki çok satırlı yapı veya fonksiyon çağrısını kapatır.
    },
    # Çok satırlı yapıdaki bu parametre veya öğeyi tanımlar.
    n_jobs=-1,
    # Çok satırlı yapıdaki bu parametre veya öğeyi tanımlar.
    error_score="raise",
# Önceki çok satırlı yapı veya fonksiyon çağrısını kapatır.
)

# `cv_summary` değişkenine bu adımda kullanılacak değeri atar.
cv_summary = pd.DataFrame(
    # Çok satırlı veri yapısını veya fonksiyon çağrısını başlatır.
    {
        # Pipeline'ın bu aşaması için gerekli Python ifadesini çalıştırır.
        "metric": [
            # Çok satırlı yapıdaki bu parametre veya öğeyi tanımlar.
            "R2",
            # Çok satırlı yapıdaki bu parametre veya öğeyi tanımlar.
            "RMSE",
            # Çok satırlı yapıdaki bu parametre veya öğeyi tanımlar.
            "MAE",
        # Önceki çok satırlı yapı veya fonksiyon çağrısını kapatır.
        ],
        # Pipeline'ın bu aşaması için gerekli Python ifadesini çalıştırır.
        "mean": [
            # Pipeline'ın bu aşaması için gerekli Python ifadesini çalıştırır.
            cv_result[
                # Pipeline'ın bu aşaması için gerekli Python ifadesini çalıştırır.
                "test_R2"
            # Önceki çok satırlı yapı veya fonksiyon çağrısını kapatır.
            ].mean(),
            # Çok satırlı veri yapısını veya fonksiyon çağrısını başlatır.
            (
                # Pipeline'ın bu aşaması için gerekli Python ifadesini çalıştırır.
                -cv_result[
                    # Pipeline'ın bu aşaması için gerekli Python ifadesini çalıştırır.
                    "test_RMSE"
                # Önceki çok satırlı yapı veya fonksiyon çağrısını kapatır.
                ]
            # Önceki çok satırlı yapı veya fonksiyon çağrısını kapatır.
            ).mean(),
            # Çok satırlı veri yapısını veya fonksiyon çağrısını başlatır.
            (
                # Pipeline'ın bu aşaması için gerekli Python ifadesini çalıştırır.
                -cv_result[
                    # Pipeline'ın bu aşaması için gerekli Python ifadesini çalıştırır.
                    "test_MAE"
                # Önceki çok satırlı yapı veya fonksiyon çağrısını kapatır.
                ]
            # Önceki çok satırlı yapı veya fonksiyon çağrısını kapatır.
            ).mean(),
        # Önceki çok satırlı yapı veya fonksiyon çağrısını kapatır.
        ],
        # Pipeline'ın bu aşaması için gerekli Python ifadesini çalıştırır.
        "std": [
            # Pipeline'ın bu aşaması için gerekli Python ifadesini çalıştırır.
            cv_result[
                # Pipeline'ın bu aşaması için gerekli Python ifadesini çalıştırır.
                "test_R2"
            # Önceki çok satırlı yapı veya fonksiyon çağrısını kapatır.
            ].std(
                # `ddof` değişkenine bu adımda kullanılacak değeri atar.
                ddof=1
            # Önceki çok satırlı yapı veya fonksiyon çağrısını kapatır.
            ),
            # Çok satırlı veri yapısını veya fonksiyon çağrısını başlatır.
            (
                # Pipeline'ın bu aşaması için gerekli Python ifadesini çalıştırır.
                -cv_result[
                    # Pipeline'ın bu aşaması için gerekli Python ifadesini çalıştırır.
                    "test_RMSE"
                # Önceki çok satırlı yapı veya fonksiyon çağrısını kapatır.
                ]
            # Önceki çok satırlı yapı veya fonksiyon çağrısını kapatır.
            ).std(
                # `ddof` değişkenine bu adımda kullanılacak değeri atar.
                ddof=1
            # Önceki çok satırlı yapı veya fonksiyon çağrısını kapatır.
            ),
            # Çok satırlı veri yapısını veya fonksiyon çağrısını başlatır.
            (
                # Pipeline'ın bu aşaması için gerekli Python ifadesini çalıştırır.
                -cv_result[
                    # Pipeline'ın bu aşaması için gerekli Python ifadesini çalıştırır.
                    "test_MAE"
                # Önceki çok satırlı yapı veya fonksiyon çağrısını kapatır.
                ]
            # Önceki çok satırlı yapı veya fonksiyon çağrısını kapatır.
            ).std(
                # `ddof` değişkenine bu adımda kullanılacak değeri atar.
                ddof=1
            # Önceki çok satırlı yapı veya fonksiyon çağrısını kapatır.
            ),
        # Önceki çok satırlı yapı veya fonksiyon çağrısını kapatır.
        ],
    # Önceki çok satırlı yapı veya fonksiyon çağrısını kapatır.
    }
# Önceki çok satırlı yapı veya fonksiyon çağrısını kapatır.
)

# `cv_summary_display` değişkenine bu adımda kullanılacak değeri atar.
cv_summary_display = (
    # Pipeline'ın bu aşaması için gerekli Python ifadesini çalıştırır.
    cv_summary.copy()
# Önceki çok satırlı yapı veya fonksiyon çağrısını kapatır.
)

# Pipeline'ın bu aşaması için gerekli Python ifadesini çalıştırır.
cv_summary_display[
    # Çok satırlı veri yapısını veya fonksiyon çağrısını başlatır.
    [
        # Çok satırlı yapıdaki bu parametre veya öğeyi tanımlar.
        "mean",
        # Çok satırlı yapıdaki bu parametre veya öğeyi tanımlar.
        "std",
    # Önceki çok satırlı yapı veya fonksiyon çağrısını kapatır.
    ]
# Önceki çok satırlı yapı veya fonksiyon çağrısını kapatır.
] = (
    # Pipeline'ın bu aşaması için gerekli Python ifadesini çalıştırır.
    cv_summary_display[
        # Çok satırlı veri yapısını veya fonksiyon çağrısını başlatır.
        [
            # Çok satırlı yapıdaki bu parametre veya öğeyi tanımlar.
            "mean",
            # Çok satırlı yapıdaki bu parametre veya öğeyi tanımlar.
            "std",
        # Önceki çok satırlı yapı veya fonksiyon çağrısını kapatır.
        ]
    # Önceki çok satırlı yapı veya fonksiyon çağrısını kapatır.
    ]
    # Sayısal sonuçları okunabilir ondalık basamak sayısına yuvarlar.
    .round(4)
# Önceki çok satırlı yapı veya fonksiyon çağrısını kapatır.
)

# Tabloyu notebook ekranında okunabilir biçimde gösterir.
display(
    # Pipeline'ın bu aşaması için gerekli Python ifadesini çalıştırır.
    metrics_display
# Önceki çok satırlı yapı veya fonksiyon çağrısını kapatır.
)

# Tabloyu notebook ekranında okunabilir biçimde gösterir.
display(
    # Pipeline'ın bu aşaması için gerekli Python ifadesini çalıştırır.
    cv_summary_display
# Önceki çok satırlı yapı veya fonksiyon çağrısını kapatır.
)

## 9 — Lider model tahmin tabloları ve grafikler

Bu bölüm LazyPredict lider modelinin train ve test tahmin tablolarını hazırlar.

İlk grafik gerçek pIC50 (`pStandard`) ile tahmin edilen pIC50 değerlerini karşılaştırır. İdeal durumda noktalar kesikli `y = x` çizgisine yakın yer alır.

İkinci grafik yalnızca test setindeki tahmin artıklarını gösterir. Artıkların sıfır çevresinde dengeli dağılması, belirgin sistematik tahmin yanlılığının bulunmadığını düşündürür.


In [ ]:
# `train_output` değişkenine bu adımda kullanılacak değeri atar.
train_output = feature_df.iloc[
    # Pipeline'ın bu aşaması için gerekli Python ifadesini çalıştırır.
    train_indices
# Önceki çok satırlı yapı veya fonksiyon çağrısını kapatır.
][metadata_columns].copy()

# Pipeline'ın bu aşaması için gerekli Python ifadesini çalıştırır.
train_output["actual_pIC50"] = (
    # Pipeline'ın bu aşaması için gerekli Python ifadesini çalıştırır.
    y_train.to_numpy()
# Önceki çok satırlı yapı veya fonksiyon çağrısını kapatır.
)

# Pipeline'ın bu aşaması için gerekli Python ifadesini çalıştırır.
train_output["predicted_pIC50"] = (
    # Pipeline'ın bu aşaması için gerekli Python ifadesini çalıştırır.
    train_prediction
# Önceki çok satırlı yapı veya fonksiyon çağrısını kapatır.
)

# Pipeline'ın bu aşaması için gerekli Python ifadesini çalıştırır.
train_output["residual"] = (
    # Pipeline'ın bu aşaması için gerekli Python ifadesini çalıştırır.
    y_train.to_numpy()
    # Pipeline'ın bu aşaması için gerekli Python ifadesini çalıştırır.
    - train_prediction
# Önceki çok satırlı yapı veya fonksiyon çağrısını kapatır.
)

# Pipeline'ın bu aşaması için gerekli Python ifadesini çalıştırır.
train_output["source_index"] = train_indices

# `test_output` değişkenine bu adımda kullanılacak değeri atar.
test_output = feature_df.iloc[
    # Pipeline'ın bu aşaması için gerekli Python ifadesini çalıştırır.
    test_indices
# Önceki çok satırlı yapı veya fonksiyon çağrısını kapatır.
][metadata_columns].copy()

# Pipeline'ın bu aşaması için gerekli Python ifadesini çalıştırır.
test_output["actual_pIC50"] = (
    # Pipeline'ın bu aşaması için gerekli Python ifadesini çalıştırır.
    y_test.to_numpy()
# Önceki çok satırlı yapı veya fonksiyon çağrısını kapatır.
)

# Pipeline'ın bu aşaması için gerekli Python ifadesini çalıştırır.
test_output["predicted_pIC50"] = (
    # Pipeline'ın bu aşaması için gerekli Python ifadesini çalıştırır.
    test_prediction
# Önceki çok satırlı yapı veya fonksiyon çağrısını kapatır.
)

# Pipeline'ın bu aşaması için gerekli Python ifadesini çalıştırır.
test_output["residual"] = (
    # Pipeline'ın bu aşaması için gerekli Python ifadesini çalıştırır.
    y_test.to_numpy()
    # Pipeline'ın bu aşaması için gerekli Python ifadesini çalıştırır.
    - test_prediction
# Önceki çok satırlı yapı veya fonksiyon çağrısını kapatır.
)

# Pipeline'ın bu aşaması için gerekli Python ifadesini çalıştırır.
test_output["source_index"] = test_indices

# `safe_leader_name` değişkenine bu adımda kullanılacak değeri atar.
safe_leader_name = re.sub(
    # Çok satırlı yapıdaki bu parametre veya öğeyi tanımlar.
    r"[^A-Za-z0-9_.-]+",
    # Çok satırlı yapıdaki bu parametre veya öğeyi tanımlar.
    "_",
    # Çok satırlı yapıdaki bu parametre veya öğeyi tanımlar.
    leader_model_name,
# Önceki çok satırlı yapı veya fonksiyon çağrısını kapatır.
)

# `actual_prediction_plot_path` değişkenine bu adımda kullanılacak değeri atar.
actual_prediction_plot_path = (
    # Pipeline'ın bu aşaması için gerekli Python ifadesini çalıştırır.
    DRIVE_DIR
    # Pipeline'ın bu aşaması için gerekli Python ifadesini çalıştırır.
    / (
        # Pipeline'ın bu aşaması için gerekli Python ifadesini çalıştırır.
        f"{TARGET_ID}_{safe_leader_name}_"
        # Pipeline'ın bu aşaması için gerekli Python ifadesini çalıştırır.
        "true_vs_predicted_pIC50.png"
    # Önceki çok satırlı yapı veya fonksiyon çağrısını kapatır.
    )
# Önceki çok satırlı yapı veya fonksiyon çağrısını kapatır.
)

# `residual_plot_path` değişkenine bu adımda kullanılacak değeri atar.
residual_plot_path = (
    # Pipeline'ın bu aşaması için gerekli Python ifadesini çalıştırır.
    DRIVE_DIR
    # Pipeline'ın bu aşaması için gerekli Python ifadesini çalıştırır.
    / (
        # Pipeline'ın bu aşaması için gerekli Python ifadesini çalıştırır.
        f"{TARGET_ID}_{safe_leader_name}_"
        # Pipeline'ın bu aşaması için gerekli Python ifadesini çalıştırır.
        "test_residuals.png"
    # Önceki çok satırlı yapı veya fonksiyon çağrısını kapatır.
    )
# Önceki çok satırlı yapı veya fonksiyon çağrısını kapatır.
)

# Yeni bir grafik alanı oluşturur.
plt.figure(figsize=(7, 7))

# Değerleri saçılım grafiği üzerinde gösterir.
plt.scatter(
    # Çok satırlı yapıdaki bu parametre veya öğeyi tanımlar.
    y_train,
    # Çok satırlı yapıdaki bu parametre veya öğeyi tanımlar.
    train_prediction,
    # Çok satırlı yapıdaki bu parametre veya öğeyi tanımlar.
    alpha=0.50,
    # Çok satırlı yapıdaki bu parametre veya öğeyi tanımlar.
    label="Train",
# Önceki çok satırlı yapı veya fonksiyon çağrısını kapatır.
)

# Değerleri saçılım grafiği üzerinde gösterir.
plt.scatter(
    # Çok satırlı yapıdaki bu parametre veya öğeyi tanımlar.
    y_test,
    # Çok satırlı yapıdaki bu parametre veya öğeyi tanımlar.
    test_prediction,
    # Çok satırlı yapıdaki bu parametre veya öğeyi tanımlar.
    alpha=0.80,
    # Çok satırlı yapıdaki bu parametre veya öğeyi tanımlar.
    label="Test",
# Önceki çok satırlı yapı veya fonksiyon çağrısını kapatır.
)

# `axis_min` değişkenine bu adımda kullanılacak değeri atar.
axis_min = min(
    # Çok satırlı yapıdaki bu parametre veya öğeyi tanımlar.
    y.min(),
    # Çok satırlı yapıdaki bu parametre veya öğeyi tanımlar.
    train_prediction.min(),
    # Çok satırlı yapıdaki bu parametre veya öğeyi tanımlar.
    test_prediction.min(),
# Önceki çok satırlı yapı veya fonksiyon çağrısını kapatır.
)

# `axis_max` değişkenine bu adımda kullanılacak değeri atar.
axis_max = max(
    # Çok satırlı yapıdaki bu parametre veya öğeyi tanımlar.
    y.max(),
    # Çok satırlı yapıdaki bu parametre veya öğeyi tanımlar.
    train_prediction.max(),
    # Çok satırlı yapıdaki bu parametre veya öğeyi tanımlar.
    test_prediction.max(),
# Önceki çok satırlı yapı veya fonksiyon çağrısını kapatır.
)

# Pipeline'ın bu aşaması için gerekli Python ifadesini çalıştırır.
plt.plot(
    # Çok satırlı veri yapısını veya fonksiyon çağrısını başlatır.
    [axis_min, axis_max],
    # Çok satırlı veri yapısını veya fonksiyon çağrısını başlatır.
    [axis_min, axis_max],
    # Çok satırlı yapıdaki bu parametre veya öğeyi tanımlar.
    linestyle="--",
# Önceki çok satırlı yapı veya fonksiyon çağrısını kapatır.
)

# Grafiğin yatay eksen etiketini tanımlar.
plt.xlabel("True pIC50 (pStandard)")
# Grafiğin dikey eksen etiketini tanımlar.
plt.ylabel("Predicted pIC50 (pStandard)")

# Grafiğe açıklayıcı başlık ekler.
plt.title(
    # Pipeline'ın bu aşaması için gerekli Python ifadesini çalıştırır.
    f"{TARGET_ID} — {leader_model_name}"
# Önceki çok satırlı yapı veya fonksiyon çağrısını kapatır.
)

# Grafikteki veri gruplarının açıklamasını gösterir.
plt.legend()
# Grafik elemanlarının üst üste gelmesini azaltır.
plt.tight_layout()

# Grafiği yüksek çözünürlüklü dosya olarak kaydeder.
plt.savefig(
    # Çok satırlı yapıdaki bu parametre veya öğeyi tanımlar.
    actual_prediction_plot_path,
    # Çok satırlı yapıdaki bu parametre veya öğeyi tanımlar.
    dpi=300,
    # Çok satırlı yapıdaki bu parametre veya öğeyi tanımlar.
    bbox_inches="tight",
# Önceki çok satırlı yapı veya fonksiyon çağrısını kapatır.
)

# Hazırlanan grafiği notebook ekranında gösterir.
plt.show()

# `test_residuals` değişkenine bu adımda kullanılacak değeri atar.
test_residuals = (
    # Pipeline'ın bu aşaması için gerekli Python ifadesini çalıştırır.
    y_test.to_numpy()
    # Pipeline'ın bu aşaması için gerekli Python ifadesini çalıştırır.
    - test_prediction
# Önceki çok satırlı yapı veya fonksiyon çağrısını kapatır.
)

# Yeni bir grafik alanı oluşturur.
plt.figure(figsize=(8, 6))

# Değerleri saçılım grafiği üzerinde gösterir.
plt.scatter(
    # Çok satırlı yapıdaki bu parametre veya öğeyi tanımlar.
    test_prediction,
    # Çok satırlı yapıdaki bu parametre veya öğeyi tanımlar.
    test_residuals,
    # Çok satırlı yapıdaki bu parametre veya öğeyi tanımlar.
    alpha=0.80,
# Önceki çok satırlı yapı veya fonksiyon çağrısını kapatır.
)

# Grafiğe yorumlamayı kolaylaştıran referans çizgisi ekler.
plt.axhline(
    # Çok satırlı yapıdaki bu parametre veya öğeyi tanımlar.
    0.0,
    # Çok satırlı yapıdaki bu parametre veya öğeyi tanımlar.
    linestyle="--",
# Önceki çok satırlı yapı veya fonksiyon çağrısını kapatır.
)

# Grafiğin yatay eksen etiketini tanımlar.
plt.xlabel("Predicted pIC50 (pStandard)")
# Grafiğin dikey eksen etiketini tanımlar.
plt.ylabel("Residual: true − predicted")

# Grafiğe açıklayıcı başlık ekler.
plt.title(
    # Pipeline'ın bu aşaması için gerekli Python ifadesini çalıştırır.
    f"{TARGET_ID} — {leader_model_name} test residuals"
# Önceki çok satırlı yapı veya fonksiyon çağrısını kapatır.
)

# Grafik elemanlarının üst üste gelmesini azaltır.
plt.tight_layout()

# Grafiği yüksek çözünürlüklü dosya olarak kaydeder.
plt.savefig(
    # Çok satırlı yapıdaki bu parametre veya öğeyi tanımlar.
    residual_plot_path,
    # Çok satırlı yapıdaki bu parametre veya öğeyi tanımlar.
    dpi=300,
    # Çok satırlı yapıdaki bu parametre veya öğeyi tanımlar.
    bbox_inches="tight",
# Önceki çok satırlı yapı veya fonksiyon çağrısını kapatır.
)

# Hazırlanan grafiği notebook ekranında gösterir.
plt.show()

## 10 — LazyPredict modelleri, lider model ve bundle kayıtları

Bu bölüm LazyPredict benchmarkında başarıyla eğitilmiş bütün model nesnelerini ayrı `.joblib` dosyaları olarak Google Drive'a kaydeder.

Dosya adlarında sorun oluşturabilecek karakterler güvenli alt çizgi karakterine çevrilir. Her model için kayıt durumu ve olası hata mesajı bir registry CSV dosyasına yazılır.

Lider model ayrıca sabit bir dosya adıyla kaydedilir. Notebook 04 bu lider model dosyasını bundle üzerinden yükler; başka bir model eğitmez.


In [ ]:
# `lazy_models_directory` değişkenine bu adımda kullanılacak değeri atar.
lazy_models_directory = (
    # Pipeline'ın bu aşaması için gerekli Python ifadesini çalıştırır.
    DRIVE_DIR / LAZY_MODELS_FOLDERNAME
# Önceki çok satırlı yapı veya fonksiyon çağrısını kapatır.
)

# Gerekli klasörü mevcut değilse oluşturur.
lazy_models_directory.mkdir(
    # Çok satırlı yapıdaki bu parametre veya öğeyi tanımlar.
    parents=True,
    # Çok satırlı yapıdaki bu parametre veya öğeyi tanımlar.
    exist_ok=True,
# Önceki çok satırlı yapı veya fonksiyon çağrısını kapatır.
)

# `model_registry_rows` değişkenine bu adımda kullanılacak değeri atar.
model_registry_rows = []

# Koleksiyondaki öğeleri sırayla işler.
for rank, model_name in enumerate(
    # Çok satırlı yapıdaki bu parametre veya öğeyi tanımlar.
    lazy_models.index,
    # Çok satırlı yapıdaki bu parametre veya öğeyi tanımlar.
    start=1,
# Önceki çok satırlı yapı veya fonksiyon çağrısını kapatır.
):
    # `model_name` değişkenine bu adımda kullanılacak değeri atar.
    model_name = str(model_name)

    # `safe_model_name` değişkenine bu adımda kullanılacak değeri atar.
    safe_model_name = re.sub(
        # Çok satırlı yapıdaki bu parametre veya öğeyi tanımlar.
        r"[^A-Za-z0-9_.-]+",
        # Çok satırlı yapıdaki bu parametre veya öğeyi tanımlar.
        "_",
        # Çok satırlı yapıdaki bu parametre veya öğeyi tanımlar.
        model_name,
    # Önceki çok satırlı yapı veya fonksiyon çağrısını kapatır.
    )

    # `model_filename` değişkenine bu adımda kullanılacak değeri atar.
    model_filename = (
        # Pipeline'ın bu aşaması için gerekli Python ifadesini çalıştırır.
        f"{rank:02d}_{safe_model_name}.joblib"
    # Önceki çok satırlı yapı veya fonksiyon çağrısını kapatır.
    )

    # `model_path` değişkenine bu adımda kullanılacak değeri atar.
    model_path = (
        # Pipeline'ın bu aşaması için gerekli Python ifadesini çalıştırır.
        lazy_models_directory
        # Pipeline'ın bu aşaması için gerekli Python ifadesini çalıştırır.
        / model_filename
    # Önceki çok satırlı yapı veya fonksiyon çağrısını kapatır.
    )

    # `model_object` değişkenine bu adımda kullanılacak değeri atar.
    model_object = lazy_model_dictionary.get(
        # Pipeline'ın bu aşaması için gerekli Python ifadesini çalıştırır.
        model_name
    # Önceki çok satırlı yapı veya fonksiyon çağrısını kapatır.
    )

    # Belirtilen koşul doğruysa aşağıdaki kod bloğunu çalıştırır.
    if model_object is None:
        # Pipeline'ın bu aşaması için gerekli Python ifadesini çalıştırır.
        model_registry_rows.append(
            # Çok satırlı veri yapısını veya fonksiyon çağrısını başlatır.
            {
                # Çok satırlı yapıdaki bu parametre veya öğeyi tanımlar.
                "rank": rank,
                # Çok satırlı yapıdaki bu parametre veya öğeyi tanımlar.
                "model_name": model_name,
                # Çok satırlı yapıdaki bu parametre veya öğeyi tanımlar.
                "filename": "",
                # Çok satırlı yapıdaki bu parametre veya öğeyi tanımlar.
                "saved": False,
                # Çok satırlı yapıdaki bu parametre veya öğeyi tanımlar.
                "error": "Model nesnesi bulunamadı.",
            # Önceki çok satırlı yapı veya fonksiyon çağrısını kapatır.
            }
        # Önceki çok satırlı yapı veya fonksiyon çağrısını kapatır.
        )

        # Pipeline'ın bu aşaması için gerekli Python ifadesini çalıştırır.
        continue

    # Hata oluşturabilecek işlemi güvenli biçimde dener.
    try:
        # Eğitilmiş model nesnesini Google Drive'a kaydeder.
        joblib.dump(
            # Çok satırlı yapıdaki bu parametre veya öğeyi tanımlar.
            model_object,
            # Çok satırlı yapıdaki bu parametre veya öğeyi tanımlar.
            model_path,
        # Önceki çok satırlı yapı veya fonksiyon çağrısını kapatır.
        )

        # Pipeline'ın bu aşaması için gerekli Python ifadesini çalıştırır.
        model_registry_rows.append(
            # Çok satırlı veri yapısını veya fonksiyon çağrısını başlatır.
            {
                # Çok satırlı yapıdaki bu parametre veya öğeyi tanımlar.
                "rank": rank,
                # Çok satırlı yapıdaki bu parametre veya öğeyi tanımlar.
                "model_name": model_name,
                # Çok satırlı yapıdaki bu parametre veya öğeyi tanımlar.
                "filename": model_filename,
                # Çok satırlı yapıdaki bu parametre veya öğeyi tanımlar.
                "saved": True,
                # Çok satırlı yapıdaki bu parametre veya öğeyi tanımlar.
                "error": "",
            # Önceki çok satırlı yapı veya fonksiyon çağrısını kapatır.
            }
        # Önceki çok satırlı yapı veya fonksiyon çağrısını kapatır.
        )

    # Belirtilen hata oluşursa kontrollü alternatif işlemi çalıştırır.
    except Exception as model_save_error:
        # Pipeline'ın bu aşaması için gerekli Python ifadesini çalıştırır.
        model_registry_rows.append(
            # Çok satırlı veri yapısını veya fonksiyon çağrısını başlatır.
            {
                # Çok satırlı yapıdaki bu parametre veya öğeyi tanımlar.
                "rank": rank,
                # Çok satırlı yapıdaki bu parametre veya öğeyi tanımlar.
                "model_name": model_name,
                # Çok satırlı yapıdaki bu parametre veya öğeyi tanımlar.
                "filename": model_filename,
                # Çok satırlı yapıdaki bu parametre veya öğeyi tanımlar.
                "saved": False,
                # Çok satırlı yapıdaki bu parametre veya öğeyi tanımlar.
                "error": str(model_save_error),
            # Önceki çok satırlı yapı veya fonksiyon çağrısını kapatır.
            }
        # Önceki çok satırlı yapı veya fonksiyon çağrısını kapatır.
        )

# `leader_model_path` değişkenine bu adımda kullanılacak değeri atar.
leader_model_path = (
    # Pipeline'ın bu aşaması için gerekli Python ifadesini çalıştırır.
    DRIVE_DIR / LEADER_MODEL_FILENAME
# Önceki çok satırlı yapı veya fonksiyon çağrısını kapatır.
)

# `imputer_path` değişkenine bu adımda kullanılacak değeri atar.
imputer_path = (
    # Pipeline'ın bu aşaması için gerekli Python ifadesini çalıştırır.
    DRIVE_DIR / IMPUTER_FILENAME
# Önceki çok satırlı yapı veya fonksiyon çağrısını kapatır.
)

# `bundle_path` değişkenine bu adımda kullanılacak değeri atar.
bundle_path = (
    # Pipeline'ın bu aşaması için gerekli Python ifadesini çalıştırır.
    DRIVE_DIR / BUNDLE_FILENAME
# Önceki çok satırlı yapı veya fonksiyon çağrısını kapatır.
)

# `model_registry_path` değişkenine bu adımda kullanılacak değeri atar.
model_registry_path = (
    # Pipeline'ın bu aşaması için gerekli Python ifadesini çalıştırır.
    DRIVE_DIR
    # Pipeline'ın bu aşaması için gerekli Python ifadesini çalıştırır.
    / LAZY_MODEL_REGISTRY_FILENAME
# Önceki çok satırlı yapı veya fonksiyon çağrısını kapatır.
)

# Eğitilmiş model nesnesini Google Drive'a kaydeder.
joblib.dump(
    # Çok satırlı yapıdaki bu parametre veya öğeyi tanımlar.
    leader_model,
    # Çok satırlı yapıdaki bu parametre veya öğeyi tanımlar.
    leader_model_path,
# Önceki çok satırlı yapı veya fonksiyon çağrısını kapatır.
)

# Eğitilmiş model nesnesini Google Drive'a kaydeder.
joblib.dump(
    # Çok satırlı yapıdaki bu parametre veya öğeyi tanımlar.
    imputer,
    # Çok satırlı yapıdaki bu parametre veya öğeyi tanımlar.
    imputer_path,
# Önceki çok satırlı yapı veya fonksiyon çağrısını kapatır.
)

# `model_registry` değişkenine bu adımda kullanılacak değeri atar.
model_registry = pd.DataFrame(
    # Pipeline'ın bu aşaması için gerekli Python ifadesini çalıştırır.
    model_registry_rows
# Önceki çok satırlı yapı veya fonksiyon çağrısını kapatır.
)

# Hazırlanan tabloyu CSV dosyası olarak kaydeder.
model_registry.to_csv(
    # Çok satırlı yapıdaki bu parametre veya öğeyi tanımlar.
    model_registry_path,
    # Çok satırlı yapıdaki bu parametre veya öğeyi tanımlar.
    index=False,
# Önceki çok satırlı yapı veya fonksiyon çağrısını kapatır.
)

# `bundle` değişkenine bu adımda kullanılacak değeri atar.
bundle = {
    # Çok satırlı yapıdaki bu parametre veya öğeyi tanımlar.
    "target_id": TARGET_ID,
    # Çok satırlı yapıdaki bu parametre veya öğeyi tanımlar.
    "feature_filename": FEATURE_FILENAME,
    # Çok satırlı yapıdaki bu parametre veya öğeyi tanımlar.
    "target_column": TARGET_COLUMN,
    # Çok satırlı yapıdaki bu parametre veya öğeyi tanımlar.
    "metadata_columns": metadata_columns,
    # Çok satırlı yapıdaki bu parametre veya öğeyi tanımlar.
    "feature_names": feature_columns,
    # Çok satırlı yapıdaki bu parametre veya öğeyi tanımlar.
    "train_indices": train_indices.tolist(),
    # Çok satırlı yapıdaki bu parametre veya öğeyi tanımlar.
    "test_indices": test_indices.tolist(),
    # Çok satırlı yapıdaki bu parametre veya öğeyi tanımlar.
    "random_state": RANDOM_STATE,
    # Çok satırlı yapıdaki bu parametre veya öğeyi tanımlar.
    "test_size": TEST_SIZE,
    # Çok satırlı yapıdaki bu parametre veya öğeyi tanımlar.
    "leader_model_name": leader_model_name,
    # Çok satırlı yapıdaki bu parametre veya öğeyi tanımlar.
    "selected_model_name": leader_model_name,
    # Çok satırlı yapıdaki bu parametre veya öğeyi tanımlar.
    "leader_model_parameters": leader_model_parameters,
    # Çok satırlı yapıdaki bu parametre veya öğeyi tanımlar.
    "leader_model_filename": LEADER_MODEL_FILENAME,
    # Çok satırlı yapıdaki bu parametre veya öğeyi tanımlar.
    "model_filename": LEADER_MODEL_FILENAME,
    # Çok satırlı yapıdaki bu parametre veya öğeyi tanımlar.
    "imputer_filename": IMPUTER_FILENAME,
    # Çok satırlı yapıdaki bu parametre veya öğeyi tanımlar.
    "lazy_models_foldername": LAZY_MODELS_FOLDERNAME,
    # Pipeline'ın bu aşaması için gerekli Python ifadesini çalıştırır.
    "lazy_model_registry_filename": (
        # Pipeline'ın bu aşaması için gerekli Python ifadesini çalıştırır.
        LAZY_MODEL_REGISTRY_FILENAME
    # Önceki çok satırlı yapı veya fonksiyon çağrısını kapatır.
    ),
# Önceki çok satırlı yapı veya fonksiyon çağrısını kapatır.
}

# Metin veya JSON içeriğini belirtilen dosyaya kaydeder.
bundle_path.write_text(
    # Pipeline'ın bu aşaması için gerekli Python ifadesini çalıştırır.
    json.dumps(
        # Çok satırlı yapıdaki bu parametre veya öğeyi tanımlar.
        bundle,
        # Çok satırlı yapıdaki bu parametre veya öğeyi tanımlar.
        ensure_ascii=False,
        # Çok satırlı yapıdaki bu parametre veya öğeyi tanımlar.
        indent=2,
    # Önceki çok satırlı yapı veya fonksiyon çağrısını kapatır.
    ),
    # Çok satırlı yapıdaki bu parametre veya öğeyi tanımlar.
    encoding="utf-8",
# Önceki çok satırlı yapı veya fonksiyon çağrısını kapatır.
)

# Hazırlanan tabloyu CSV dosyası olarak kaydeder.
lazy_models.reset_index().to_csv(
    # Pipeline'ın bu aşaması için gerekli Python ifadesini çalıştırır.
    DRIVE_DIR
    # Çok satırlı yapıdaki bu parametre veya öğeyi tanımlar.
    / f"{TARGET_ID}_LazyPredict_results.csv",
    # Çok satırlı yapıdaki bu parametre veya öğeyi tanımlar.
    index=False,
    # Çok satırlı yapıdaki bu parametre veya öğeyi tanımlar.
    float_format="%.6f",
# Önceki çok satırlı yapı veya fonksiyon çağrısını kapatır.
)

# Hazırlanan tabloyu CSV dosyası olarak kaydeder.
lazy_predictions.to_csv(
    # Pipeline'ın bu aşaması için gerekli Python ifadesini çalıştırır.
    DRIVE_DIR
    # Çok satırlı yapıdaki bu parametre veya öğeyi tanımlar.
    / f"{TARGET_ID}_LazyPredict_predictions.csv",
    # Çok satırlı yapıdaki bu parametre veya öğeyi tanımlar.
    index=True,
    # Çok satırlı yapıdaki bu parametre veya öğeyi tanımlar.
    float_format="%.6f",
# Önceki çok satırlı yapı veya fonksiyon çağrısını kapatır.
)

# Hazırlanan tabloyu CSV dosyası olarak kaydeder.
metrics.to_csv(
    # Pipeline'ın bu aşaması için gerekli Python ifadesini çalıştırır.
    DRIVE_DIR
    # Çok satırlı yapıdaki bu parametre veya öğeyi tanımlar.
    / f"{TARGET_ID}_leader_model_metrics.csv",
    # Çok satırlı yapıdaki bu parametre veya öğeyi tanımlar.
    index=False,
    # Çok satırlı yapıdaki bu parametre veya öğeyi tanımlar.
    float_format="%.6f",
# Önceki çok satırlı yapı veya fonksiyon çağrısını kapatır.
)

# Hazırlanan tabloyu CSV dosyası olarak kaydeder.
cv_summary.to_csv(
    # Pipeline'ın bu aşaması için gerekli Python ifadesini çalıştırır.
    DRIVE_DIR
    # Çok satırlı yapıdaki bu parametre veya öğeyi tanımlar.
    / f"{TARGET_ID}_leader_model_5CV.csv",
    # Çok satırlı yapıdaki bu parametre veya öğeyi tanımlar.
    index=False,
    # Çok satırlı yapıdaki bu parametre veya öğeyi tanımlar.
    float_format="%.6f",
# Önceki çok satırlı yapı veya fonksiyon çağrısını kapatır.
)

# Hazırlanan tabloyu CSV dosyası olarak kaydeder.
train_output.to_csv(
    # Pipeline'ın bu aşaması için gerekli Python ifadesini çalıştırır.
    DRIVE_DIR
    # Çok satırlı yapıdaki bu parametre veya öğeyi tanımlar.
    / f"{TARGET_ID}_leader_train_predictions.csv",
    # Çok satırlı yapıdaki bu parametre veya öğeyi tanımlar.
    index=False,
    # Çok satırlı yapıdaki bu parametre veya öğeyi tanımlar.
    float_format="%.6f",
# Önceki çok satırlı yapı veya fonksiyon çağrısını kapatır.
)

# Hazırlanan tabloyu CSV dosyası olarak kaydeder.
test_output.to_csv(
    # Pipeline'ın bu aşaması için gerekli Python ifadesini çalıştırır.
    DRIVE_DIR
    # Çok satırlı yapıdaki bu parametre veya öğeyi tanımlar.
    / f"{TARGET_ID}_leader_test_predictions.csv",
    # Çok satırlı yapıdaki bu parametre veya öğeyi tanımlar.
    index=False,
    # Çok satırlı yapıdaki bu parametre veya öğeyi tanımlar.
    float_format="%.6f",
# Önceki çok satırlı yapı veya fonksiyon çağrısını kapatır.
)

# Kontrol veya sonuç bilgisini ekrana yazdırır.
print("Lider model:", leader_model_path)
# Kontrol veya sonuç bilgisini ekrana yazdırır.
print("Kaydedilen model klasörü:", lazy_models_directory)
# Kontrol veya sonuç bilgisini ekrana yazdırır.
print("Model registry:", model_registry_path)
# Kontrol veya sonuç bilgisini ekrana yazdırır.
print("Bundle:", bundle_path)